# Fundamentales desde SEC EDGAR

Baja los estados financieros que la SEC publica, **con la fecha en que se presentaron**, y reporta qué porcentaje del universo tiene cada métrica de verdad.

## Por qué EDGAR y no un proveedor

El bloque `valuation_carry` pesa 10–12% del compuesto y su propio texto admite que corre con *proxies*: no hay P/E ni EV/EBITDA en ninguna parte del modelo. En una corrida real dos de sus tres métricas salieron `UNAVAILABLE from Yahoo`.

Cualquier proveedor de múltiplos arregla eso. **Ninguno arregla el problema de abajo.** Un vendor te da el número de hoy, ya corregido, y un backtest alimentado con datos restatados está recibiendo información que nadie tenía entonces — el IC que salga de ahí está inflado por construcción.

EDGAR no tiene ese problema porque no es un proveedor: **es el archivo**. Cada dato trae el `filed` de la presentación que lo trajo, y las versiones sucesivas conviven como entradas separadas. Filtrar `filed <= fecha` reconstruye lo que se sabía ese día por construcción, no por promesa de nadie.

Gratis, sin llave, y es la fuente primaria de la que los vendors revenden.

## El histórico viene desde la primera corrida

A diferencia de los precios, aquí no hay que acumular nada. Una sola llamada devuelve **todo lo que la empresa ha reportado bajo XBRL**, que son unos diez años. No esperas: bajas y ya lo tienes.

## Lo que este notebook NO hace

No calcula ratios y **no toca el modelo de scoring**. Baja hechos, los mapea a conceptos declarados y reporta cobertura. Un P/E necesita casar un fundamental con un precio alineando las dos fechas, y esa es una decisión aparte que todavía no está tomada.

El entregable es el **reporte de cobertura**: con él se decide si vale la pena construir el bloque fundamental, sobre número medido y no sobre esperanza.


## 1 · Motor


In [ ]:
# El paquete screener/ del repo, embebido. Mismo tarball y misma
# verificacion que el notebook principal: un motor solo, no dos.
import base64, gzip, hashlib, io, sys, tarfile

ENGINE_SHA256 = "9329fc52c870b6f56e2d53473aff9317b3d41b6b670f235f7e9f64ec721ccc28"
ENGINE_B64 = (
    "..."
)

raw = gzip.decompress(base64.b64decode(ENGINE_B64))
digest = hashlib.sha256(raw).hexdigest()
assert digest == ENGINE_SHA256, f'motor alterado: {digest}'
with tarfile.open(fileobj=io.BytesIO(raw)) as tar:
    tar.extractall('/content')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from screener import edgar
print(f'motor verificado  sha256={ENGINE_SHA256[:16]}...')
print(f'{len(edgar.CONCEPTOS)} conceptos declarados')


## 2 · Parámetros

**`CONTACTO` no es opcional.** La SEC exige identificarse con un correo real y **bloquea por IP** a quien no lo hace. No es burocracia: es la condición de uso de un servicio gratuito.

**Empieza con `LIMITE = 3`.** Si tres nombres bajan bien, el resto es lo mismo repetido. Lanzar 400 peticiones para descubrir que el `User-Agent` estaba mal es la forma cara de aprenderlo.

**Si agregamos conceptos nuevos al modelo**, lo ya bajado no los tiene y la celda lo avisa: la descarga es incremental por existencia de archivo, así que un nombre viejo se salta y su cobertura para la métrica nueva sale en **cero sin ser cero**. Marca `REBAJAR_TODO` cuando eso pase.

**Guardar en Drive es lo que hace la descarga reanudable.** El disco de Colab desaparece cuando el runtime se recicla; con el almacén en Drive, volver a correr esto retoma donde quedó en vez de empezar de cero.

Si el montaje falla — pasa por un popup bloqueado, por cookies de terceros desactivadas, o por cancelarlo — **la celda sigue** y el almacén cae en el disco de Colab. En ese caso la sección 6 deja de ser opcional: baja el zip antes de cerrar la sesión.

Para arreglar el montaje: permite ventanas emergentes para `colab.research.google.com`, habilita cookies de terceros, y si insiste prueba *Entorno de ejecución → Desconectar y eliminar el entorno* antes de reintentar.


In [ ]:
# @markdown ### Identificación ante la SEC (obligatoria)
CONTACTO = "CCI Puesto de Bolsa tucorreo@dominio.com"  # @param {type:"string"}

# @markdown ### Qué bajar
UNIVERSO = "sp500"  # @param ["sp500", "acciones", "ndx", "djia", "lista"]
TICKERS_PERSONALIZADOS = "AAPL,MSFT,NVDA"  # @param {type:"string"}
# @markdown Cuántos nombres como máximo. 0 = todos. **Empieza en 3.**
LIMITE = 3  # @param {type:"integer"}

# @markdown ### Dónde guardar
GUARDAR_EN_DRIVE = True  # @param {type:"boolean"}
# @markdown Sin Drive el almacén se pierde al reciclarse el runtime y
# @markdown la próxima corrida vuelve a bajar todo.
REBAJAR_TODO = False  # @param {type:"boolean"}
# @markdown Marca solo para refrescar lo que ya está en disco.
REMAPEAR = False  # @param {type:"boolean"}
# @markdown Vuelve a pedir el mapa ticker→CIK sin rebajar los
# @markdown fundamentales: dos peticiones, no doscientas. Márcalo si un
# @markdown nombre vigente sale «sin CIK».

from pathlib import Path

edgar.user_agent(CONTACTO)   # falla aqui si falta el correo

# El montaje de Drive falla por cosas que no dependen de este codigo:
# un popup de autenticacion bloqueado, cookies de terceros desactivadas,
# o simplemente cancelarlo. Que eso mate la celda dejaria la corrida sin
# empezar por un problema de permisos del navegador, asi que degrada y lo
# dice: el almacen cae en el disco de Colab y la seccion 6 se vuelve
# obligatoria en vez de opcional.
DESTINO = None
if GUARDAR_EN_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DESTINO = Path('/content/drive/MyDrive/CCI_Fundamentales')
    except Exception as _exc:
        print(f'AVISO: no se pudo montar Drive '
              f'({type(_exc).__name__}: {_exc}).')
        print('       El almacen va al disco de Colab, que se recicla.')
        print('       BAJA EL ZIP DE LA SECCION 6 antes de cerrar, o')
        print('       reintenta el montaje y vuelve a correr esta celda.')
        print()

if DESTINO is None:
    DESTINO = Path('/content/fundamentales')
DESTINO.mkdir(parents=True, exist_ok=True)

if UNIVERSO == 'lista':
    TICKERS = [t.strip().upper() for t in TICKERS_PERSONALIZADOS.split(',')
               if t.strip()]
else:
    from screener.universe import all_index_members
    _grupos = {'sp500': ('SP500',), 'acciones': ('SP500', 'NDX', 'DJIA'),
               'ndx': ('NDX',), 'djia': ('DJIA',)}
    _miembros = all_index_members()
    _fuera = set()
    for _clave in _grupos[UNIVERSO]:
        _fuera |= set(_miembros.get(_clave, frozenset()))
    # Yahoo usa guion donde la SEC usa punto (BRK-B vs BRK.B); el mapa de
    # la SEC trae guion.
    TICKERS = sorted(t.replace('.', '-') for t in _fuera)

if LIMITE:
    TICKERS = TICKERS[:LIMITE]

_nuevos = edgar.conceptos_desactualizados(DESTINO)
if _nuevos and not REBAJAR_TODO:
    print('AVISO: el almacen se escribio con una lista de conceptos '
          'anterior.')
    print(f'       Estos {len(_nuevos)} son posteriores y NO estan en '
          'lo ya bajado:')
    print(f'       {", ".join(_nuevos)}')
    print('       Su cobertura saldra en CERO SIN SER CERO. Marca '
          'REBAJAR_TODO')
    print('       para rehacerlo, o ignoralo si esas metricas no te '
          'importan aun.')
    print()

print(f'{len(TICKERS)} nombre(s) -> {DESTINO}')
print(f'Contacto: {CONTACTO}')
_ya = len(list(DESTINO.glob('[!_]*.csv')))
if _ya:
    print(f'Ya en el almacen: {_ya} nombre(s). '
          + ('Se rebajan todos.' if REBAJAR_TODO else 'No se vuelven a pedir.'))


## 3 · Descarga

Una petición por emisor, espaciadas para no pasar del tope de la SEC. Un `companyfacts` de una empresa grande pesa 10–15 MB, así que el universo completo son varios minutos y unos pocos GB de tráfico — de los que se guarda menos del 5%, que es lo que declara `CONCEPTOS`.

Si el runtime se cae a mitad, vuelve a correr esta celda: lo que ya está en disco no se vuelve a pedir.

Un emisor que falle no tumba la corrida. Los extranjeros que presentan 20-F suelen traer menos etiquetas, y algunos ninguna de las que conocemos; salen listados al final en vez de rellenarse.

### Si un nombre vigente sale «sin CIK»

El ticker→CIK sale de tres archivos oficiales — `company_tickers.json`, `company_tickers_exchange.json` y `ticker.txt` — y la SEC dice de ellos que los actualiza pero **no garantiza su exactitud ni su alcance**. Un registrante vivo puede faltar en los tres. Ya pasó: en una corrida con las tres listas completas faltaban ocho miembros del S&P 500.

Faltar, entonces, no prueba nada sobre el emisor. Búscalo en [CIK Lookup](https://www.sec.gov/search-filings/cik-lookup) y anota una línea en `_ciks_manuales.csv`, dentro de la carpeta del almacén:

```
ticker,cik,por_que
AVB,915912,verificado en EDGAR 2026-09
```

Se lee en cada corrida, también con el mapa cacheado, y **solo rellena huecos**: nunca contradice a la SEC en silencio. El nombre que la SEC tiene para ese CIK queda impreso y en `_emisores.csv` — míralo, porque un CIK equivocado no da error: da los estados financieros de otra empresa con tu ticker encima.


In [ ]:
import time

limitador = edgar.Limitador()
_cache_mapa = DESTINO / '_tickers.json'
mapa = edgar.load_ticker_map(contacto=CONTACTO, cache=_cache_mapa,
                             limitador=limitador, refrescar=REMAPEAR)
# De que lista salio cada cosa. Sin esto, 'sin CIK' no se puede leer:
# no se sabe si falta el emisor o si falta el mapa.
_fuentes = edgar.fuentes_del_mapa(_cache_mapa)
_manuales = edgar.leer_overrides(DESTINO)
print(f'Mapa ticker->CIK: {len(mapa):,} emisores'
      + (' (' + ', '.join(f'{_k}={_fuentes[_k]:,}' for _k in
                          ('company_tickers', 'company_tickers_exchange',
                           'ticker_txt') if _k in _fuentes) + ')'
         if 'company_tickers' in _fuentes else ''))
if _manuales:
    print(f'  + {len(_manuales)} CIK a mano en '
          f'{edgar.ARCHIVO_OVERRIDES}: ' + ', '.join(sorted(_manuales)))
if len(mapa) < edgar.MIN_EMISORES:
    print(f'  AVISO: son menos de {edgar.MIN_EMISORES:,}. El mapa esta '
          'incompleto y lo que salga sin CIK no prueba nada.')
print()

etiquetas, ok, sin_cik, fallaron, saltados = [], [], [], [], []
# Que nombre tiene la SEC para cada CIK. Es la comprobacion de que el
# CIK es el correcto: uno equivocado no da error, da los estados de
# otra empresa con nuestro ticker encima.
emisores = []
# El motivo de cada fallo, para no depender del scrollback: un nombre
# que falla dos corridas seguidas necesita diagnostico, y 'sin CIK' y
# '404' llevan a sitios distintos.
motivos = []
_t0 = time.time()

for _i, _tk in enumerate(TICKERS, 1):
    if (DESTINO / f'{_tk}.csv').exists() and not REBAJAR_TODO:
        saltados.append(_tk)
        continue

    _cik = mapa.get(_tk) or mapa.get(_tk.replace('-', '.'))
    if not _cik:
        sin_cik.append(_tk)
        motivos.append({'ticker': _tk, 'motivo': 'sin CIK',
                        'detalle': edgar.detalle_sin_cik(_fuentes)})
        print(f'  [{_i:3d}/{len(TICKERS)}] {_tk:6s} sin CIK en la SEC')
        continue

    try:
        _payload = edgar.company_facts(_cik, contacto=CONTACTO,
                                       limitador=limitador)
        _hechos, _elegidas = edgar.extract_facts(_payload, _tk)
    except Exception as _exc:
        fallaron.append(_tk)
        motivos.append({'ticker': _tk, 'motivo': type(_exc).__name__,
                        'detalle': f'CIK {_cik}: {_exc}'})
        print(f'  [{_i:3d}/{len(TICKERS)}] {_tk:6s} '
              f'FALLO {type(_exc).__name__}: {_exc}')
        continue

    if not _hechos:
        fallaron.append(_tk)
        motivos.append({'ticker': _tk, 'motivo': 'sin etiquetas',
                        'detalle': f'CIK {_cik}: companyfacts respondio '
                                   'sin ninguna etiqueta de CONCEPTOS'})
        print(f'  [{_i:3d}/{len(TICKERS)}] {_tk:6s} '
              'sin ninguna etiqueta conocida (¿emisor extranjero?)')
        continue

    edgar.escribir_hechos(DESTINO, _tk, _hechos)
    ok.append(_tk)
    emisores.append({'ticker': _tk, 'cik': _cik,
                     'entidad': _payload.get('entityName', ''),
                     'fuente': 'a mano' if _tk in _manuales else 'SEC'})
    for _m, _e in _elegidas.items():
        etiquetas.append({'ticker': _tk, 'metrica': _m, 'etiqueta': _e})
    print(f'  [{_i:3d}/{len(TICKERS)}] {_tk:6s} {len(_hechos):5d} hechos, '
          f'{len(_elegidas)}/{len(edgar.CONCEPTOS)} metricas')

print(f'\n{len(ok)} bajados, {len(saltados)} ya estaban, '
      f'{len(sin_cik)} sin CIK, {len(fallaron)} fallaron '
      f'({time.time() - _t0:.0f}s)')
import pandas as _pd
# _fallos.csv describe LA ULTIMA corrida, asi que se reescribe siempre
# que la corrida haya intentado algo, vacio incluido. Dejarlo puesto es
# peor que no escribirlo: la corrida que arreglo los ocho nombres los
# dejaria ahi, con el diagnostico viejo, al lado de una cobertura del
# 100%. Dos archivos que se contradicen y ninguna forma de saber cual
# es el de hoy.
if ok or sin_cik or fallaron:
    _pd.DataFrame(motivos, columns=['ticker', 'motivo', 'detalle']).to_csv(
        DESTINO / '_fallos.csv', index=False)
    print(f'  Motivos en _fallos.csv ({len(motivos)} nombre(s))'
          if motivos else '  _fallos.csv queda vacio: no fallo ninguno.')
if ok or REBAJAR_TODO:
    edgar.escribir_manifiesto(DESTINO)
if emisores:
    # Acumulativo: la descarga es incremental, asi que una corrida que
    # baja ocho nombres no puede borrar el registro de los otros 272.
    _ruta_em = DESTINO / '_emisores.csv'
    _previos = (_pd.read_csv(_ruta_em) if _ruta_em.exists() and not
                REBAJAR_TODO else _pd.DataFrame())
    _pd.concat([_previos, _pd.DataFrame(emisores)], ignore_index=True) \
       .drop_duplicates('ticker', keep='last').to_csv(_ruta_em,
                                                      index=False)
    _a_mano = [_e for _e in emisores if _e['fuente'] == 'a mano']
    if _a_mano:
        print('\nCIK puestos a mano — verifica que el nombre sea el '
              'que esperas:')
        for _e in _a_mano:
            print(f"  {_e['ticker']:6s} {_e['cik']}  {_e['entidad']}")
if sin_cik:
    print(f'  Sin CIK: {", ".join(sin_cik[:20])}')
    print(f'           {edgar.detalle_sin_cik(_fuentes)}')
    _plantilla = edgar.plantilla_overrides(DESTINO, sin_cik)
    if _plantilla:
        print(f'  Te deje {_plantilla} con esos nombres y el CIK en '
              'blanco: llenalo y vuelve a correr esta celda.')
if fallaron:
    print(f'  Fallaron: {", ".join(fallaron[:20])}')


## 4 · Cobertura — el entregable

Ésta es la tabla que decide si vale la pena construir el bloque fundamental. Con 60% de cobertura, un z-score transversal compara a los que reportaron contra un hueco, y eso no es una medición.

Una métrica ausente sale en **0%, no desaparece**. Desaparecer se lee como *no aplica*; cero se lee como *no lo tenemos*.

Mira la columna `etiquetas_usadas`. Si dice 3, significa que tres etiquetas XBRL distintas trajeron la misma idea en el mismo universo — XBRL es un vocabulario, no un esquema, y sin la tabla de prioridad de `CONCEPTOS` una parte de tus emisores habría quedado sin ese dato.


In [ ]:
import pandas as pd

if etiquetas:
    _modo = 'w' if REBAJAR_TODO or not (DESTINO / '_etiquetas.csv').exists() else 'a'
    pd.DataFrame(etiquetas).to_csv(DESTINO / '_etiquetas.csv',
                                   mode=_modo, index=False,
                                   header=(_modo == 'w'))

hechos = edgar.leer_hechos(DESTINO, TICKERS)
if hechos.empty:
    print('No hay nada en el almacen todavia.')
else:
    cobertura = edgar.coverage_report(hechos, TICKERS)
    cobertura.to_csv(DESTINO / '_cobertura.csv', index=False)

    historia = edgar.historia_por_ticker(hechos)
    print(f'{len(historia)} nombre(s), {len(hechos):,} hechos, '
          f'de {historia["desde"].min()} a {historia["hasta"].max()}')
    print(f'Periodos distintos por nombre: mediana '
          f'{historia["periodos"].median():.0f}')


def _semaforo(v):
    """Rojo bajo 60%, ambar hasta 85%, verde arriba.

    A mano y no con background_gradient porque ese exige matplotlib, y
    una dependencia mas es una forma mas de que la celda reviente en la
    maquina de otro.
    """
    if v is None or not isinstance(v, (int, float)):
        return ''
    if v < 0.60:
        return 'background-color:#7F1D1D;color:#FFFFFF'
    if v < 0.85:
        return 'background-color:#78350F;color:#FFFFFF'
    return 'background-color:#14532D;color:#FFFFFF'

display(cobertura[['metrica', 'cobertura', 'con_dato', 'sin_dato',
                   'etiquetas_usadas', 'etiqueta_principal']].style
        .format({'cobertura': '{:.1%}'})
        .map(_semaforo, subset=['cobertura'])
        .hide(axis='index')
        .set_caption('Cobertura por metrica'))


## 5 · El point-in-time, visto

La razón de ser de todo esto, en una tabla.

Cada fila es un período que se reportó **más de una vez con cifras distintas**: la empresa presentó un número y después lo corrigió. Un proveedor te habría dado directamente el corregido, y un backtest alimentado con él estaría viendo algo que en su momento nadie vio.

Si esta tabla sale vacía no es que el mecanismo falle: es que en el universo que bajaste nadie corrigió nada por encima del 2%. Con cientos de nombres y diez años, salen.


In [ ]:
if not hechos.empty:
    rest = edgar.restatements(hechos)
    print(f'{len(rest)} periodo(s) reportados dos veces con cambio >= 2%')
    if not rest.empty:
        rest.to_csv(DESTINO / '_restatements.csv', index=False)
        display(rest.head(15).style
                .format({'cambio': '{:+.1%}', 'primero': '{:,.0f}',
                         'ultimo': '{:,.0f}'})
                .hide(axis='index')
                .set_caption('Lo que un proveedor te habria dado ya corregido'))


### La misma pregunta, dos fechas

`as_of(fecha)` devuelve la última versión de cada período presentada **en o antes** de ese día. Cambia `FECHA_CORTE` y mira cómo cambia el número: eso es exactamente lo que un backtest honesto necesita y lo que ningún vendor te puede dar.


In [ ]:
FECHA_CORTE = "2024-06-30"  # @param {type:"date"}
METRICA = "activos"  # @param ["ingresos", "utilidad_neta", "activos", "patrimonio", "ebit", "efectivo", "flujo_operativo", "capex", "eps_diluido"]

if not hechos.empty:
    _entonces = edgar.as_of(hechos, FECHA_CORTE, metricas=[METRICA])
    _hoy = edgar.as_of(hechos, None, metricas=[METRICA])
    print(f'{METRICA}: {len(_entonces)} hecho(s) conocibles al {FECHA_CORTE}, '
          f'{len(_hoy)} conocidos hoy')
    if not _entonces.empty:
        display(_entonces[['ticker', 'fin', 'valor', 'filed', 'forma',
                           'etiqueta']].head(20))


## 6 · Llevarte el almacén

Si guardaste en Drive ya está a salvo y esta celda sobra. Si no, **bájalo antes de cerrar**: el disco de Colab se recicla y con él se va la descarga entera.


In [ ]:
import shutil

_zip = shutil.make_archive('/content/fundamentales', 'zip', DESTINO)
print(f'{_zip}  ({Path(_zip).stat().st_size / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(_zip)
except Exception as _exc:
    print(f'Fuera de Colab, no hay descarga automatica: {_exc}')


---

## Qué hacer con la tabla de cobertura

Si `ingresos`, `patrimonio` y `activos` salen **por encima de 90%**, el bloque fundamental es viable y la Fase 3 tiene sentido.

Si salen **cerca de 60%**, el z-score transversal estaría comparando a los que reportaron contra un hueco. Eso no se arregla rellenando con el promedio del sector — daría un número con apariencia de medición — sino ampliando el mapeo de `CONCEPTOS` o aceptando que el bloque solo aplica a un subconjunto declarado del universo.

**Un aviso sobre lo que esto todavía no arregla:** calibrar el IC sobre el universo actual lo **sobreestima**, porque la lista de nombres es una foto estática con sesgo de supervivencia — las empresas que quebraron no están. EDGAR sí las tiene. Hasta que el universo se arregle, un IC medido será mejor que el 0.08 supuesto sin ser todavía el número bueno.
